<a href="https://colab.research.google.com/github/darshita27-cmd/Music-Recommender/blob/main/InvisibleDrum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import pygame
import time
import os

# ---------------- SOUND SETUP ----------------
pygame.mixer.init()
sound_path = os.path.join(os.getcwd(), "sounds")

SOUNDS = {
    "kick": pygame.mixer.Sound(os.path.join(sound_path, r"D:\College work\pycharm\PythonProject1\sounds\KICKS\07_Kick_01_SP.wav")),
    "snare": pygame.mixer.Sound(os.path.join(sound_path, r"D:\College work\pycharm\PythonProject1\sounds\SNARES\07_Snare_02_SP.wav")),
    "hihat_closed": pygame.mixer.Sound(os.path.join(sound_path, r"D:\College work\pycharm\PythonProject1\sounds\HIHATS\07_Hats_03_SP.wav")),
    "hihat_open": pygame.mixer.Sound(os.path.join(sound_path, r"D:\College work\pycharm\PythonProject1\sounds\HIHATS\ASZ_OPENHAT.wav")),
    "tom1": pygame.mixer.Sound(os.path.join(sound_path, r"D:\College work\pycharm\PythonProject1\sounds\TOMS\Tomm1.wav")),
    "tom2": pygame.mixer.Sound(os.path.join(sound_path, r"D:\College work\pycharm\PythonProject1\sounds\TOMS\Tomm2.wav")),
    "tom3": pygame.mixer.Sound(os.path.join(sound_path, r"D:\College work\pycharm\PythonProject1\sounds\TOMS\Tomm3.wav")),
    "crash": pygame.mixer.Sound(os.path.join(sound_path, r"D:\College work\pycharm\PythonProject1\sounds\CYMBALS\CRASHH.wav")),
    "ride": pygame.mixer.Sound(os.path.join(sound_path, r"D:\College work\pycharm\PythonProject1\sounds\CYMBALS\Ride_3.wav")),
}


# ---------------- MOVENET SETUP ----------------
model = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")
def movenet_infer(img):
    img = tf.image.resize_with_pad(tf.expand_dims(img, axis=0), 192, 192)
    input_img = tf.cast(img, dtype=tf.int32)
    outputs = model.signatures['serving_default'](input_img)
    keypoints = outputs['output_0'].numpy()[0, 0, :, :2]  # (17, 2)
    return keypoints

# ---------------- CAMERA SETUP ----------------
cap = cv2.VideoCapture(0)
prev_y = {"lw": 0, "rw": 0, "la": 0, "ra": 0}
last_hit = {"lw": 0, "rw": 0, "la": 0, "ra": 0}
SENSITIVITY = 20
COOLDOWN = 0.25

print("🥁 Invisible Drum Ready! Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    keypoints = movenet_infer(img_rgb)
    h, w, _ = frame.shape
    coords = [(int(x * w), int(y * h)) for x, y in keypoints]

    left_wrist = coords[9]
    right_wrist = coords[10]
    left_ankle = coords[15]
    right_ankle = coords[16]

    now = time.time()

    lw_vel = prev_y["lw"] - left_wrist[1]
    rw_vel = prev_y["rw"] - right_wrist[1]
    la_vel = prev_y["la"] - left_ankle[1]
    ra_vel = prev_y["ra"] - right_ankle[1]

    # ----- LEFT HAND -----
    if lw_vel > SENSITIVITY and now - last_hit["lw"] > COOLDOWN:
        if left_wrist[0] < w * 0.33:
            SOUNDS["crash"].play()  # far left zone
        elif left_wrist[1] < h * 0.4:
            SOUNDS["tom1"].play()   # upper left
        else:
            SOUNDS["snare"].play()  # center left
        last_hit["lw"] = now

    # ----- RIGHT HAND -----
    if rw_vel > SENSITIVITY and now - last_hit["rw"] > COOLDOWN:
        if right_wrist[0] > w * 0.66:
            SOUNDS["ride"].play()   # far right
        elif right_wrist[1] < h * 0.4:
            SOUNDS["tom2"].play()   # upper right
        else:
            SOUNDS["hihat_closed"].play()  # mid right
        last_hit["rw"] = now

    # ----- LEGS -----
    if (la_vel > SENSITIVITY or ra_vel > SENSITIVITY) and now - last_hit["la"] > 0.5:
        SOUNDS["kick"].play()
        last_hit["la"] = now

    # Draw landmarks
    for x, y in coords:
        cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

    cv2.imshow("Invisible Drum Kit 🥁", frame)

    prev_y.update({
        "lw": left_wrist[1],
        "rw": right_wrist[1],
        "la": left_ankle[1],
        "ra": right_ankle[1],
    })

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()